# KAE GPU Runner (Kaggle)

Ноутбук-обёртка для RFC 0022 Runner. Управляется Manager'ом через `kaggle kernels push`.

Что делает:
1. Клонирует репозиторий BookAssembler
2. Ставит зависимости (fastapi, uvicorn, transformers, qwen-vl-utils, cloudflared)
3. Регистрирует loaders (Qwen2.5-VL по умолчанию)
4. Запускает `python -m src.agents.runner` в фоне
5. Поднимает cloudflared-туннель, публичный URL шлётся Manager'у через `/runner/announce`

**Kaggle Secrets** (Settings → Add-ons → Secrets):
- `KAE_MANAGER_URL` — куда слать announce (например `https://kae-manager.example`)
- `KAE_RUNNER_TOKEN` — общий Bearer-токен Manager↔Runner
- (опц.) `HUGGINGFACE_TOKEN`

In [ ]:
# 1. Клонируем репу и ставим зависимости.
!git clone --depth=1 https://github.com/4stm4/BookAssembler.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install fastapi 'uvicorn[standard]' transformers==4.49.0 qwen-vl-utils accelerate
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. Секреты → env.
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ['KAE_MANAGER_URL'] = sec.get_secret('KAE_MANAGER_URL')
os.environ['KAE_RUNNER_TOKEN'] = sec.get_secret('KAE_RUNNER_TOKEN')
os.environ['KAE_RUNNER_PORT'] = '5005'
os.environ['KAE_RUNNER_WARMUP_TASKS'] = 'vision'
os.environ['KAE_RUNNER_IDLE_TIMEOUT'] = '900'

In [ ]:
# 3. Регистрируем Qwen2.5-VL как loader vision/table/formula.
#    (Пока loader'ы для реальных моделей ещё не добавлены в репу — используется EchoLoader.
#    Этот раздел активируется, когда в src/agents/runner/loaders/qwen_vl.py появится реализация.)
# from src.agents.runner.loaders.qwen_vl import Qwen25VLLoader
# LOADERS = [Qwen25VLLoader(max_pixels=1280*28*28)]
LOADERS = None  # → EchoLoader (see src/agents/runner/app.py)

In [ ]:
# 4. Поднимаем cloudflared, узнаём публичный URL, экспортируем в env.
import re, subprocess, itertools, os
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5005'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in itertools.islice(proc.stdout, 300):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
assert url, 'cloudflared did not report a public URL'
os.environ['KAE_RUNNER_PUBLIC_URL'] = url
print('Runner will announce as:', url)

In [ ]:
# 5. Запускаем Runner (foreground). Он сам:
#    - прогреет declared warmup_tasks
#    - POST /runner/announce Manager'у
#    - завершится через KAE_RUNNER_IDLE_TIMEOUT сек простоя
import sys
sys.path.insert(0, '/kaggle/working/repo')
!KAE_RUNNER_HOST=0.0.0.0 python -m src.agents.runner